# 11 · Compose catalogs and reusable Parquet

Join registered files, save a derived relation, and reuse it through the same Kelvo boundary.

Open this notebook in Google Colab with a CPU runtime, or use a Linux x86-64
Jupyter environment with Python 3.11 or newer. No database account is required.
Run the cells in order. The first cell downloads a verified Kelvo preview binary
and installs pinned Python readers; it does not compile Kelvo.

Every analytical SQL query below runs through the **Kelvo executable**. PyArrow
reads its Arrow output, and pandas/matplotlib format small results. The pinned
DuckDB driver materializes execution before delivering Arrow batches; delivery
backpressure is not a worker memory bound.


**Data:** [NYC TLC trip records and taxi-zone lookup](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page).
This lesson uses the January 2024 yellow-taxi Parquet file. See the
[NYC Open Data terms of use](https://opendata.cityofnewyork.us/overview/#termsofuse)
and TLC's data dictionary and accuracy notice; do not treat public availability
as a warranty or apply the notebook's code license to the data.
The original Parquet download is roughly 50 MB and contains roughly three million
rows. The helper checks the pinned file digest. Queries use explicit result
limits; a small result does not imply a small source scan.


[All notebooks](https://github.com/SYNEHQ/kelvo-go/blob/cargo/notebooks/README.md)
| [Kelvo documentation](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/README.md)

## Set up a private lab


In [ ]:
# Default setup installs pinned Python packages and verifies the reviewed helper.
# Operators validating a checkout can explicitly set KELVO_NOTEBOOK_SKIP_INSTALL=1
# and KELVO_NOTEBOOK_HELPER=/absolute/path/to/kelvo_notebooks.py before starting.
import hashlib
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request

KELVO_NOTEBOOK_REF = "eacb7e2aedd8020575ecfede53db819112c40199"
KELVO_HELPER_SHA256 = "86696a66a3c903eb8f18d529fcef055e8bee72234da324b8776529138436d9a9"
if os.environ.get("KELVO_NOTEBOOK_SKIP_INSTALL") != "1":
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "pyarrow==25.0.1", "pandas==2.3.3", "matplotlib==3.10.7", "PyYAML==6.0.3",
    ], check=True)

helper_directory = None
local_helper = os.environ.get("KELVO_NOTEBOOK_HELPER")
if local_helper:
    helper_path = Path(local_helper).expanduser().resolve(strict=True)
else:
    helper_url = (
        "https://raw.githubusercontent.com/SYNEHQ/kelvo-go/"
        + KELVO_NOTEBOOK_REF + "/notebooks/kelvo_notebooks.py"
    )
    request = urllib.request.Request(helper_url, headers={"User-Agent": "kelvo-notebooks"})
    with urllib.request.urlopen(request, timeout=60) as response:
        final_url = response.geturl()
        if urllib.parse.urlsplit(final_url).scheme != "https" or final_url != helper_url:
            raise ValueError("Helper download did not retain its pinned HTTPS URL")
        payload = response.read(1024 * 1024 + 1)
    if len(payload) > 1024 * 1024:
        raise ValueError("Unexpected helper size")
    if hashlib.sha256(payload).hexdigest() != KELVO_HELPER_SHA256:
        raise ValueError("Helper checksum mismatch")
    helper_directory = Path(tempfile.mkdtemp(prefix="kelvo-notebook-helper-"))
    helper_path = helper_directory / "kelvo_notebooks.py"
    helper_path.write_bytes(payload)

spec = importlib.util.spec_from_file_location("kelvo_notebooks", helper_path)
helper = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helper
spec.loader.exec_module(helper)
from kelvo_notebooks import Lab, remote_query

lab = Lab.create()
print("Kelvo workspace:", lab.root)


In [ ]:
import pyarrow.parquet as pq
taxi, zones = lab.taxi(), lab.zones()
config = lab.catalog(sources=[
    {"id": "trips", "type": "parquet", "path": str(taxi)},
    {"id": "zones", "type": "csv", "path": str(zones)},
])
demand = lab.query(config, """
    SELECT "PULocationID" AS zone_id, COUNT(*)::BIGINT AS trips
    FROM trips
    WHERE tpep_pickup_datetime >= TIMESTAMP '2024-01-01'
      AND tpep_pickup_datetime < TIMESTAMP '2024-01-08'
    GROUP BY 1 ORDER BY zone_id
""", sources=["trips"], max_rows=1000).table()
derived = lab.root / "weekly-demand.parquet"
pq.write_table(demand, derived, compression="snappy")
assert pq.ParquetFile(derived).metadata.num_rows == demand.num_rows


## Register the derived relation

This is an explicit notebook-created Parquet artifact. It has no refresh
schedule, freshness contract or atomic generation lifecycle. The next lesson
uses Kelvo acceleration for those controls.


In [ ]:
reusable = lab.catalog(sources=[
    {"id": "demand", "type": "parquet", "path": str(derived)},
    {"id": "zones", "type": "csv", "path": str(zones)},
])
joined = lab.query(reusable, """
    SELECT d.zone_id, COALESCE(z."Zone", 'Unmapped') AS zone,
           COALESCE(z."Borough", 'Unmapped') AS borough, d.trips
    FROM demand d LEFT JOIN zones z ON d.zone_id = z."LocationID"
    ORDER BY d.zone_id
""", sources=["demand", "zones"], max_rows=1000).table()
assert joined.num_rows == demand.num_rows
assert joined["zone_id"].to_pylist() == demand["zone_id"].to_pylist()
assert joined["trips"].to_pylist() == demand["trips"].to_pylist()
display(joined.to_pandas().sort_values("trips", ascending=False).head(15))


## Select only the catalog entries a query needs

The request must explicitly include each source. The following request omits
the `zones` source and must fail rather than resolving an unselected catalog
entry.


In [ ]:
excluded_output = lab.root / "unselected-source.arrow"
excluded = lab.command([
    "query", "--config", str(reusable), "--sources", "demand",
    "--sql", "SELECT COUNT(*) FROM zones", "--out", str(excluded_output),
], check=False)
assert excluded.returncode != 0
assert not excluded_output.exists()
print("The unselected source was unavailable.")


This lesson validates CSV/Parquet federation in embedded DuckDB. It does not
exercise live network adapters, remote pushdown or cross-database consistency.
To connect hosted databases, follow the
[native federation guide](https://github.com/SYNEHQ/kelvo-go/blob/cargo/docs/federation.md)
and its source-specific capabilities and limits.


## Clean up

Run this cell after inspecting or saving your results. Re-run the setup and data
cells if you want to restart the lesson after cleanup.


In [ ]:
# Close local processes and delete this lesson's private temporary workspace.
# Downloaded release/data caches, if configured by the helper, remain reusable.
lab.close()
if helper_directory is not None:
    shutil.rmtree(helper_directory, ignore_errors=True)
